# 🐍⚡ FastAPI en 30 minutos — Pokedex Web con Login

Clase práctica y sencilla para aprender **FastAPI** construyendo una mini web de Pokémon que:

- Consume la **[PokeAPI](https://pokeapi.co/)** para traer datos reales de Pokémon.
- Tiene un **login simple** (usuario/contraseña) con cookies.
- Muestra una **página web** (HTML) con el Pokémon del usuario.

Todo se ejecuta **dentro de este notebook**, sin salir a la terminal.

## 🗓️ Agenda (30 min aprox.)

| Tiempo | Tema |
|---|---|
| 0-5 min | ¿Qué es FastAPI? + instalación |
| 5-10 min | Primera app "Hello World" y cómo correrla en el notebook |
| 10-15 min | Endpoint que consulta la PokeAPI |
| 15-20 min | Página web en HTML con el Pokémon |
| 20-27 min | Login simple con cookies + página protegida |
| 27-30 min | Repaso y siguientes pasos |


## 1. ¿Qué es FastAPI?

**FastAPI** es un framework de Python para crear APIs (y también páginas web) de forma muy rápida.

Ventajas clave:
- Es **muy fácil de escribir**: con pocas líneas tenés un servidor funcionando.
- Usa las anotaciones de tipos de Python (`str`, `int`, etc.) para **validar datos automáticamente**.
- Genera **documentación automática** en `/docs`.
- Es súper rápido (por eso el nombre).

Hoy vamos a usarlo para construir una mini "Pokedex" web.


In [2]:
# Instalamos las librerías que necesitamos
# fastapi -> el framework web
# uvicorn -> el servidor que ejecuta la app
# requests -> para llamar a la PokeAPI
# nest_asyncio -> nos permite correr el servidor DENTRO del notebook
# python-multipart -> necesario para leer formularios (login)

!py -m pip install fastapi uvicorn requests nest-asyncio python-multipart --quiet
print("✅ Listo, dependencias instaladas")


✅ Listo, dependencias instaladas


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Nuestra primera app: "Hello Pokémon"

Una app de FastAPI arranca creando un objeto `FastAPI()`. Después, cada endpoint (URL)
se define con un decorador como `@app.get("/algo")`.


In [3]:
from fastapi import FastAPI

app = FastAPI(title="Pokedex API")

@app.get("/")
def home():
    return {"mensaje": "¡Bienvenido a la Pokedex! Visitá /docs para ver la documentación."}


### ¿Cómo lo corremos si estamos en un notebook?

Normalmente FastAPI se corre desde la terminal con `uvicorn archivo:app`.
Como estamos en un notebook, vamos a levantar el servidor en un **hilo (thread) en segundo plano**,
así el notebook sigue libre para que sigamos escribiendo código y probando.

👉 Truco: en FastAPI podemos **agregar endpoints nuevos incluso después de arrancar el servidor**,
así que lo prendemos una sola vez y después iremos sumando funcionalidades en las próximas celdas,
probando en el momento.


In [4]:
import nest_asyncio
import uvicorn
import threading
import time

nest_asyncio.apply()  # permite correr un servidor async dentro del notebook

def levantar_servidor(app, port=8000):
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)
    hilo = threading.Thread(target=server.run, daemon=True)
    hilo.start()
    time.sleep(1)  # le damos un segundo para que arranque
    print(f"🚀 Servidor corriendo en http://127.0.0.1:{port}")
    print(f"📚 Documentación interactiva en http://127.0.0.1:{port}/docs")
    return server

server = levantar_servidor(app)


🚀 Servidor corriendo en http://127.0.0.1:8000
📚 Documentación interactiva en http://127.0.0.1:8000/docs


In [5]:
# Probamos nuestro primer endpoint desde el propio notebook
import requests

r = requests.get("http://127.0.0.1:8000/")
print(r.status_code)
r.json()


200


{'mensaje': '¡Bienvenido a la Pokedex! Visitá /docs para ver la documentación.'}

Si abrís en el navegador **http://127.0.0.1:8000/docs** vas a ver la documentación
interactiva que FastAPI genera automáticamente. Es una gran ventaja frente a otros frameworks.


## 3. Consumiendo la PokeAPI

Ahora vamos a crear un endpoint `/pokemon/{nombre}` que reciba el nombre de un Pokémon,
le pregunte a la [PokeAPI](https://pokeapi.co/) por sus datos, y devuelva lo importante
en un formato simple.

`{nombre}` es un **parámetro de ruta**: lo que el usuario escriba ahí, FastAPI se lo pasa
como argumento a la función.


In [6]:
@app.get("/pokemon/{nombre}")
def obtener_pokemon(nombre: str):
    url = f"https://pokeapi.co/api/v2/pokemon/{nombre.lower()}"
    respuesta = requests.get(url)

    if respuesta.status_code != 200:
        return {"error": f"No encontré al Pokémon '{nombre}'"}

    data = respuesta.json()
    return {
        "nombre": data["name"].capitalize(),
        "id": data["id"],
        "altura": data["height"],
        "peso": data["weight"],
        "tipos": [t["type"]["name"] for t in data["types"]],
        "imagen": data["sprites"]["front_default"],
    }


Probemos el nuevo endpoint. **No hace falta reiniciar el servidor**, FastAPI ya lo reconoce:

In [7]:
r = requests.get("http://127.0.0.1:8000/pokemon/pikachu")
r.json()


{'nombre': 'Pikachu',
 'id': 25,
 'altura': 4,
 'peso': 60,
 'tipos': ['electric'],
 'imagen': 'https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/25.png'}

In [8]:
# Probemos también con uno que no existe
r = requests.get("http://127.0.0.1:8000/pokemon/pokemon-inventado")
r.json()


{'error': "No encontré al Pokémon 'pokemon-inventado'"}

## 4. Una página web de verdad (HTML)

Hasta ahora devolvimos JSON (datos "crudos"). Para devolver una **página web**, usamos
`HTMLResponse` y devolvemos directamente el HTML como texto (con un f-string).

Vamos a crear `/pokemon-web/{nombre}` que muestra una tarjeta bonita del Pokémon.


In [10]:
from fastapi.responses import HTMLResponse

@app.get("/pokemon-web/{nombre}", response_class=HTMLResponse)
def pagina_pokemon(nombre: str):
    url = f"https://pokeapi.co/api/v2/pokemon/{nombre.lower()}"
    respuesta = requests.get(url)

    if respuesta.status_code != 200:
        return "<h1>😢 Pokémon no encontrado</h1><a href='/'>Volver</a>"

    data = respuesta.json()
    tipos = ", ".join(t["type"]["name"] for t in data["types"])
    imagen = data["sprites"]["front_default"]

    return f"""
    <html>
        <head><title>{data['name'].capitalize()}</title></head>
        <body style="font-family: sans-serif; text-align: center; margin-top: 50px;">
            <h1>{data['name'].capitalize()} (#{data['id']})</h1>
            <img src="{imagen}" width="200"/>
            <p><b>Tipo(s):</b> {tipos}</p>
            <p><b>Altura:</b> {data['height']} | <b>Peso:</b> {data['weight']}</p>
        </body>
    </html>
    """


Ahora sí, esto es una **página web real**. Podés abrirla directamente en tu navegador:

👉 http://127.0.0.1:8000/pokemon-web/charizard

También podemos "verla" desde el notebook mostrando el HTML crudo:


In [11]:
r = requests.get("http://127.0.0.1:8000/pokemon-web/charizard")
print(r.text)



    <html>
        <head><title>Charizard</title></head>
        <body style="font-family: sans-serif; text-align: center; margin-top: 50px;">
            <h1>Charizard (#6)</h1>
            <img src="https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/6.png" width="200"/>
            <p><b>Tipo(s):</b> fire, flying</p>
            <p><b>Altura:</b> 17 | <b>Peso:</b> 905</p>
        </body>
    </html>
    


In [12]:
# Bonus: mostrarla renderizada dentro del propio notebook
from IPython.display import IFrame
IFrame("http://127.0.0.1:8000/pokemon-web/charizard", width=400, height=300)


## 5. Login simple con cookies

Vamos a agregar un login **muy sencillo**, ideal para entender el concepto (⚠️ no es apto
para producción, es solo para aprender):

1. Un usuario "hardcodeado" (`entrenador` / `pikachu123`).
2. Un formulario HTML en `/login`.
3. Un endpoint `POST /login` que valida usuario y contraseña.
4. Si es correcto, guardamos una **cookie** en el navegador para "recordar" que está logueado.
5. Una página `/perfil` **protegida**: solo se ve si la cookie está presente.


In [13]:
from fastapi import Form, Request
from fastapi.responses import RedirectResponse

# "Base de datos" de usuarios muy simple (solo para la demo)
USUARIOS = {
    "entrenador": "pikachu123"
}

@app.get("/login", response_class=HTMLResponse)
def formulario_login():
    return """
    <html>
        <body style="font-family: sans-serif; text-align: center; margin-top: 80px;">
            <h1>🔑 Iniciar sesión</h1>
            <form action="/login" method="post">
                <input type="text" name="usuario" placeholder="Usuario"/><br/><br/>
                <input type="password" name="contrasena" placeholder="Contraseña"/><br/><br/>
                <button type="submit">Entrar</button>
            </form>
            <p>Pista: usuario <b>entrenador</b>, contraseña <b>pikachu123</b></p>
        </body>
    </html>
    """


@app.post("/login")
def procesar_login(usuario: str = Form(...), contrasena: str = Form(...)):
    if USUARIOS.get(usuario) == contrasena:
        # Login correcto: guardamos una cookie y redirigimos al perfil
        respuesta = RedirectResponse(url="/perfil", status_code=302)
        respuesta.set_cookie(key="usuario_logueado", value=usuario)
        return respuesta
    else:
        return HTMLResponse("<h1>❌ Usuario o contraseña incorrectos</h1><a href='/login'>Volver a intentar</a>")


@app.get("/logout")
def logout():
    respuesta = RedirectResponse(url="/login", status_code=302)
    respuesta.delete_cookie("usuario_logueado")
    return respuesta


### Página protegida: `/perfil`

Esta página lee la cookie de la petición (`Request`). Si no existe, redirige a `/login`.
Si existe, le mostramos al usuario su Pokémon favorito (elegido al azar) usando la PokeAPI.


In [14]:
import random

POKEMON_FAVORITOS = ["pikachu", "eevee", "charizard", "bulbasaur", "snorlax", "gengar"]

@app.get("/perfil", response_class=HTMLResponse)
def perfil(request: Request):
    usuario = request.cookies.get("usuario_logueado")

    if not usuario:
        return RedirectResponse(url="/login", status_code=302)

    # Elegimos un pokemon random para darle la bienvenida
    pokemon_nombre = random.choice(POKEMON_FAVORITOS)
    data = requests.get(f"https://pokeapi.co/api/v2/pokemon/{pokemon_nombre}").json()
    imagen = data["sprites"]["front_default"]

    return f"""
    <html>
        <body style="font-family: sans-serif; text-align: center; margin-top: 50px;">
            <h1>👋 ¡Hola, {usuario}!</h1>
            <p>Tu Pokémon del día es:</p>
            <h2>{data['name'].capitalize()}</h2>
            <img src="{imagen}" width="180"/><br/><br/>
            <a href="/logout">Cerrar sesión</a>
        </body>
    </html>
    """


### Probemos el flujo completo de login desde el notebook

Usamos `requests.Session()` para que las cookies se guarden automáticamente entre pedidos,
tal como lo haría un navegador.


In [15]:
sesion = requests.Session()

# 1) Intento entrar al perfil SIN loguearme -> debería mandarme a /login
r = sesion.get("http://127.0.0.1:8000/perfil")
print("Sin login ->", r.url)  # vemos que terminamos en /login


Sin login -> http://127.0.0.1:8000/login


In [16]:
# 2) Ahora hago login con usuario y contraseña correctos
r = sesion.post(
    "http://127.0.0.1:8000/login",
    data={"usuario": "entrenador", "contrasena": "pikachu123"}
)
print("Después de login ->", r.url)
print(r.text)


Después de login -> http://127.0.0.1:8000/perfil

    <html>
        <body style="font-family: sans-serif; text-align: center; margin-top: 50px;">
            <h1>👋 ¡Hola, entrenador!</h1>
            <p>Tu Pokémon del día es:</p>
            <h2>Bulbasaur</h2>
            <img src="https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/1.png" width="180"/><br/><br/>
            <a href="/logout">Cerrar sesión</a>
        </body>
    </html>
    


In [17]:
# 3) Probemos también con una contraseña incorrecta (en una sesión nueva)
otra_sesion = requests.Session()
r = otra_sesion.post(
    "http://127.0.0.1:8000/login",
    data={"usuario": "entrenador", "contrasena": "clave-mala"}
)
print(r.text)


<h1>❌ Usuario o contraseña incorrectos</h1><a href='/login'>Volver a intentar</a>


🎉 Como ven, `sesion` guardó la cookie automáticamente y por eso pudo entrar a `/perfil`.

Para probarlo "de verdad" como usuario, abrí en tu navegador:

👉 **http://127.0.0.1:8000/login**

e iniciá sesión con `entrenador` / `pikachu123`.


## 6. Repaso — ¿Qué construimos?

| Endpoint | Método | Qué hace |
|---|---|---|
| `/` | GET | Mensaje de bienvenida |
| `/pokemon/{nombre}` | GET | Devuelve datos JSON de un Pokémon (PokeAPI) |
| `/pokemon-web/{nombre}` | GET | Página HTML con la tarjeta del Pokémon |
| `/login` | GET | Formulario de login |
| `/login` | POST | Valida usuario/contraseña y crea la cookie |
| `/perfil` | GET | Página protegida, solo si hay cookie válida |
| `/logout` | GET | Borra la cookie (cierra sesión) |

### Conceptos clave que vimos
- `@app.get()` / `@app.post()` para definir endpoints.
- Parámetros de ruta: `/pokemon/{nombre}`.
- `HTMLResponse` para devolver páginas web en vez de JSON.
- `Form(...)` para leer datos de un formulario HTML.
- Cookies (`set_cookie` / `request.cookies`) para manejar sesiones simples.
- Documentación automática en `/docs`.

### 🚀 Siguientes pasos (para después de la clase)
- Guardar usuarios en una base de datos real (en vez de un diccionario).
- Usar contraseñas **hasheadas** (nunca en texto plano) con `passlib`.
- Usar **JWT** o sesiones firmadas en vez de una cookie simple, para producción.
- Separar el HTML en archivos de plantillas con **Jinja2** (`Jinja2Templates`).
- Agregar un `Pydantic` model para validar mejor las respuestas.


In [18]:
# Para apagar el servidor cuando termines la clase, podés correr:
# server.should_exit = True
print("Cuando quieras detener el servidor, ejecutá: server.should_exit = True")


Cuando quieras detener el servidor, ejecutá: server.should_exit = True
